In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

Optimización de la carga de datos en PyTorch
============================================

**Autores**: [Divyansh Khanna](https://github.com/divyanshk), [Ramanish
Singh](https://github.com/ramanishsingh)

<div style="width: 45%; float: left; padding: 20px;"><h2> Qué aprenderás</h2><ul><li>Cómo optimizar la configuración del DataLoader para maximizar el rendimiento</li><li>Buenas prácticas para <code>batch_size</code>, <code>num_workers</code> y <code>pin_memory</code></li><li>Técnicas avanzadas para solapar transferencias de datos con el cómputo en GPU</li><li>Configurar estrategias de memoria compartida y gestionar problemas con <code>/dev/shm</code></li></ul></div><div style="width: 45%; float: right; padding: 20px;"><h2> Prerrequisitos</h2><ul><li>PyTorch v2.0+</li><li>Comprensión básica del DataLoader de PyTorch</li><li>(Opcional) Una GPU compatible con CUDA para las optimizaciones específicas de GPU</li></ul></div>

Introducción
------------

La carga de datos suele ser un cuello de botella crítico en los pipelines de
aprendizaje profundo. Aunque las GPU pueden procesar lotes a gran velocidad,
una carga de datos ineficiente puede dejar el hardware costoso inactivo,
esperando el siguiente lote. Este tutorial cubre las buenas prácticas y algunas
técnicas para optimizar la configuración de carga de datos y maximizar el
rendimiento del entrenamiento.

Exploraremos los parámetros clave del DataLoader de PyTorch y ofreceremos
orientación práctica para ajustarlos a tu carga de trabajo específica. En lugar
de mostrar cada optimización de forma aislada, partiremos de un bucle de
entrenamiento base e iremos aplicando optimizaciones progresivamente, midiendo
la aceleración acumulada en cada paso.


In [2]:
# Importa el módulo time para medir tiempos de ejecución en el benchmark
import time

# Importa PyTorch y sus módulos principales
import torch
import torch.nn as nn

# DataLoader: gestiona la carga de datos en lotes con soporte para multiprocesamiento, precarga y memoria fijada.
# Dataset: clase base para definir datasets personalizados con __len__ y __getitem__
from torch.utils.data import DataLoader, Dataset

# Detecta si hay una GPU disponible; si no, usa la CPU
# Esto permite que el notebook funcione tanto con GPU como sin ella
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Fija la semilla aleatoria para asegurar la reproducibilidad de los experimentos
torch.manual_seed(42)

Using device: cuda


Creación de un dataset de ejemplo
==================================

Primero, crearemos un dataset sencillo que simula transformaciones costosas.
Esto nos ayudará a demostrar el impacto de distintas configuraciones del
DataLoader.


In [ ]:
# =============================================================================
# DATASET SINTÉTICO SIN BATCHES PARA EL BENCHMARK
#
# Se definen dos datasets: uno estándar (SyntheticDatasetNonBatched) y uno con 
# soporte para obtención por lotes (SyntheticDatasetBatched). Ambos simulan una
# transformación costosa mediante time.sleep(), lo que nos permite controlar
# artificialmente la carga computacional por muestra y visualizar el impacto
# de cada optimización del DataLoader de forma clara y reproducible.
# =============================================================================

# Dataset sintético que simula transformaciones costosas por muestra
class SyntheticDatasetNonBatched(Dataset):

    def __init__(self, size=10000, feature_dim=224, transform_delay=0.001):
        self.size = size                         # Número total de muestras del dataset
        self.feature_dim = feature_dim           # Dimensión espacial de la imagen simulada (feature_dim × feature_dim)
        self.transform_delay = transform_delay   # Retardo artificial en segundos por muestra (simula una transformación costosa)

    def __len__(self):
        # Devuelve el número total de muestras del dataset
        return self.size

    def __getitem__(self, idx):
        # Genera los datos de forma perezosa (lazy): solo cuando se solicitan,
        # evitando la preasignación de grandes bloques de memoria al inicio
        data = torch.randn(3, self.feature_dim, self.feature_dim)  # Imagen aleatoria RGB
        label = torch.randint(0, 10, (1,)).item()                  # Etiqueta aleatoria entre 0 y 9
        if self.transform_delay > 0:
            time.sleep(self.transform_delay)  # Simula el coste de una transformación real (p. ej., augmentation)
        return data, label


# =============================================================================
# DATASETS SINTÉTICO CON BATCHES PARA EL BENCHMARK
# Tiene __getitems__ PARA OBTENCIÓN VECTORIZADA POR LOTES
#
# Implementa el método __getitems__(indices), que el DataLoader puede llamar
# una sola vez con una lista de índices en lugar de N llamadas individuales
# a __getitem__. Esto permite amortizar el coste de inicialización por lote
# (p. ej., una sola consulta SQL, una sola apertura de archivo, etc.) y
# generar todos los tensores del lote en una única operación vectorizada.
# =============================================================================

# Dataset sintético igual que SyntheticDatasetNonBatched pero con __getitems__ para obtención por lotes.
class SyntheticDatasetBatched(Dataset):

    def __init__(self, size=10000, feature_dim=224, transform_delay=0.001):
        self.size = size
        self.feature_dim = feature_dim
        self.transform_delay = transform_delay

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        # Ruta estándar muestra a muestra (usada si __getitems__ no está disponible)
        data = torch.randn(3, self.feature_dim, self.feature_dim)
        label = torch.randint(0, 10, (1,)).item()
        if self.transform_delay > 0:
            time.sleep(self.transform_delay)
        return data, label

    def __getitems__(self, indices):
        """Obtiene múltiples muestras a la vez — habilita la generación vectorizada.

        En lugar de N llamadas individuales a __getitem__ (cada una con su propia
        sobrecarga), genera el lote completo en una sola operación vectorizada.
        """
        n = len(indices)
        # Generación vectorizada: una sola llamada para n muestras en lugar de n llamadas
        data = torch.randn(n, 3, self.feature_dim, self.feature_dim)
        labels = torch.randint(0, 10, (n,))
        # Simula I/O a nivel de lote: un solo sleep para todo el lote,
        # no uno por muestra (equivalente a una consulta BD para N filas)
        if self.transform_delay > 0:
            time.sleep(self.transform_delay)
        return [(data[i], labels[i].item()) for i in range(n)]

Creación de un modelo al que pasar los datasets
================================================

Modelo Conv + Transformer ligero usado como carga de cómputo GPU realista.
No está optimizado para precisión; su propósito es generar trabajo real en la
GPU para que los benchmarks reflejen condiciones de entrenamiento auténticas.

In [6]:
# =============================================================================
# MODELO: SmallTransformerModel
# =============================================================================

class SmallTransformerModel(nn.Module):

    def __init__(self):
        super().__init__()
        
        # Extractor de características convolucional: reduce la imagen (3×224×224) a (64×7×7)
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, stride=4, padding=3),  # Reduce resolución x4
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # Reduce resolución x2
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((7, 7)),  # Fuerza salida a tamaño fijo 7×7
        )
        # Encoder Transformer: procesa la secuencia de 49 parches (7×7) de 64 dimensiones
        encoder_layer = nn.TransformerEncoderLayer(d_model=64, nhead=4,
                                                   dim_feedforward=128, batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # Clasificador final: mapea la representación de 64 dims a 10 clases
        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        x = self.features(x)                        # (B, 64, 7, 7)
        B, C, H, W = x.shape
        x = x.view(B, C, H * W).permute(0, 2, 1)    # Reorganiza a secuencia: (B, 49, 64)
        x = self.transformer(x)                     # Aplica atención sobre los 49 parches: (B, 49, 64)
        x = x.mean(dim=1)                           # Agrega (media) sobre la secuencia: (B, 64)
        return self.classifier(x)                   # Proyección final a logits: (B, 10)

# Crea el modelo Conv+Transformer y lo mueve al dispositivo activo. GPU si está disponible, CPU si no.
def create_model():
    return SmallTransformerModel().to(device)


Solapamiento de la transferencia H2D con el cómputo en GPU
===========================================================

Para maximizar el rendimiento, puedes solapar las transferencias Host-a-Dispositivo
(H2D) con el cómputo en GPU. Esto garantiza que la GPU nunca esté inactiva
esperando datos.

La idea es precargar el siguiente lote en la GPU mientras se procesa el lote actual.

En resumen: 

            non_blocking libera a la CPU para que no espere;
            
            DataPrefetcher libera a la GPU para que no espere, aprovechando que
            sus dos motores (cómputo y DMA) pueden operar en paralelo.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>El DataPrefetcher muestra su mayor beneficio cuando el tiempo de transferencia
H2D se solapa de forma significativa con el cómputo en GPU. Si la carga de datos
ya es rápida, la sobrecarga de sincronización de streams puede superar el beneficio.</p>

</div>

In [ ]:
# =============================================================================
# DataPrefetcher — Solapamiento de transferencia H2D con cómputo
#
# Objetivo: mantener la GPU siempre ocupada solapando la transferencia del
# siguiente lote (CPU→GPU) con el cómputo del lote actual en GPU.
#
# Mecanismo:
#   1. Se crea un CUDA stream secundario (self.stream) separado del stream
#      principal donde corre el modelo.
#   2. preload() lanza la transferencia del próximo lote en el stream secundario
#      usando non_blocking=True → la CPU no espera a que la GPU termine de copiarlo.
#   3. En __next__(), antes de devolver el lote actual, se sincroniza el stream
#      principal con el secundario (wait_stream), garantizando que los datos
#      ya están disponibles en GPU cuando el modelo los necesite.
#   4. record_stream() registra qué stream usará cada tensor para evitar que
#      PyTorch libere la memoria antes de que la transferencia haya terminado.
#
# Resultado: mientras la GPU procesa el lote N, la CPU ya está transfiriendo
# el lote N+1 → se elimina la espera de GPU entre lotes.

""" 
Lo que hace DataPrefetcher es usar la arquitectura real de la GPU, que tiene dos
motores independientes:
    - Motor de cómputo: ejecuta kernels (forward, backward…)
    - Motor DMA: hace copias de memoria (H2D) de forma independiente

DataPrefetcher lanza que se haga la copia del batch N+1 por parte de la GPU en un
stream secundario, mientras el motor de cómputo aún está procesando el batch N.
Ambos motores trabajan en paralelo en la misma GPU. Entonces cuando el motor de
cómputo termina el batch N, la copia del batch N+1 ya está hecha (o casi) y puede
empezar de inmediato.
"""
# =============================================================================

class DataPrefetcher:
    """Precarga datos en GPU mientras se procesa el lote anterior."""

    def __init__(self, loader, device):
        self.loader = iter(loader)    # Iterador sobre el DataLoader subyacente
        self.device = device
        # Stream CUDA secundario dedicado exclusivamente a las transferencias H2D
        self.stream = torch.cuda.Stream() if torch.cuda.is_available() else None
        self.next_data = None
        self.next_labels = None
        self.preload()                # Inicia la precarga del primer lote de inmediato

    def preload(self):
        """Carga el siguiente lote desde la CPU al GPU en el stream secundario."""
        try:
            self.next_data, self.next_labels = next(self.loader)
        except StopIteration:
            # No quedan más lotes: señaliza el fin de la iteración
            self.next_data = None
            self.next_labels = None
            return

        if self.stream is not None:
            # Ejecuta la transferencia en el stream secundario para no bloquear
            # el stream principal donde corre el modelo
            with torch.cuda.stream(self.stream):
                self.next_data = self.next_data.to(self.device, non_blocking=True)
                self.next_labels = self.next_labels.to(self.device, non_blocking=True)

    def __iter__(self):
        return self

    def __next__(self):
        if self.stream is not None:
            # Sincronización: el stream principal espera a que el stream secundario
            # termine de transferir el lote antes de que el modelo lo consuma
            torch.cuda.current_stream().wait_stream(self.stream)

        data = self.next_data
        labels = self.next_labels

        if data is None:
            raise StopIteration

        if self.stream is not None:
            # Registra en qué stream se usará cada tensor: PyTorch retrasará la
            # liberación de memoria hasta que el stream principal haya terminado
            # con ellos, evitando condiciones de carrera en la gestión de memoria
            data.record_stream(torch.cuda.current_stream())
            labels.record_stream(torch.cuda.current_stream())

        # Lanza ya la precarga del siguiente lote (solapado con el cómputo actual)
        self.preload()
        return data, labels

Infraestructura compartida de entrenamiento
============================================

Para medir el impacto real de cada optimización, definimos un bucle de
entrenamiento reutilizable que acepta un DataLoader y devuelve los tiempos
de ejecución y la pérdida. Esto evita duplicar el bucle de entrenamiento
para cada benchmark.

Usamos un **dataset pequeño (512 muestras)** con un **retardo de transformación
alto (5 ms)** para garantizar que el pipeline permanezca limitado por los datos
durante todo el experimento. El dataset pequeño implica épocas cortas (16 lotes
cada una), por lo que ejecutamos muchas épocas, lo que hace visible el beneficio
de `persistent_workers` en los límites entre épocas.


In [ ]:
# =============================================================================
# FUNCIÓN DE BENCHMARK: train_and_benchmark
#
# Bucle de entrenamiento reutilizable que ejecuta múltiples épocas sobre un
# DataLoader dado y devuelve el tiempo total transcurrido y la pérdida media.
# Ejecutar varias épocas (10) con un dataset pequeño asegura muchos límites
# de época, haciendo visible el ahorro de persistent_workers.
# =============================================================================

"""
Args:
    - loader: DataLoader sobre el que iterar.
    - max_batches: Número máximo total de lotes a procesar en todas las épocas.
    - epochs: Número de épocas a iterar (reitera el loader en cada época).
    - prefetch_device: Si se indica, envuelve el loader en un DataPrefetcher en
        cada época para solapar las transferencias H2D. Los datos llegan ya en el dispositivo.

Returns:
    Tupla de (segundos_transcurridos, pérdida_media).
"""

def train_and_benchmark(loader, max_batches=160, epochs=10, prefetch_device=None):

    model = create_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # Optimizador SGD estándar
    criterion = nn.CrossEntropyLoss()                          # Pérdida de clasificación multiclase

    start_time = time.perf_counter()  # Marca de inicio de tiempo de alta resolución
    total_loss = 0.0
    num_batches = 0

    for epoch in range(epochs):
        # Si se activa el prefetching, envuelve el loader con el DataPrefetcher
        # para solapar la transferencia CPU→GPU del siguiente lote con el cómputo actual
        if prefetch_device is not None:
            data_iter = DataPrefetcher(loader, prefetch_device)
        else:
            data_iter = loader

        for data, labels in data_iter:
            if prefetch_device is None:
                # Transferencia asíncrona: la CPU no espera a que la GPU confirme la copia
                # antes de continuar con la siguiente instrucción
                data = data.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

            output = model(data)                         # Forward pass
            loss = criterion(output, labels)             # Calcula la pérdida

            optimizer.zero_grad()                        # Limpia gradientes del paso anterior
            loss.backward()                              # Backpropagation
            optimizer.step()                             # Actualiza pesos

            total_loss += loss.item()
            num_batches += 1

            if num_batches >= max_batches:
                break
        if num_batches >= max_batches:
            break

    if torch.cuda.is_available():
        # Sincronización final: asegura que todas las operaciones GPU han terminado
        # antes de detener el cronómetro, para obtener una medición precisa
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    return elapsed, total_loss / num_batches

Bucle de entrenamiento base
===========================

Nuestro punto de partida: un DataLoader sencillo sin multiprocesamiento, sin
memoria fijada y con la configuración por defecto. Esto establece el rendimiento
mínimo que mejoraremos progresivamente.


In [10]:
# =============================================================================
# DATASET:
#
# Dataset pequeño (512 muestras) con retardo alto (5 ms por muestra) para que
# el pipeline sea limitado por los datos (data-bound), haciendo visible el
# impacto de cada optimización del DataLoader.
# =============================================================================

# Dataset de benchmark: pequeño y con alta latencia por muestra para que el
# cuello de botella esté siempre en la carga de datos, no en el cómputo GPU
benchmark_dataset = SyntheticDatasetNonBatched(size=512, feature_dim=224,
                                               transform_delay=0.005)

baseline_loader = DataLoader(
    benchmark_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

print("\n=== Progressive Optimization Results ===")
print("\nBaseline (num_workers=0, pin_memory=False):")
baseline_time, baseline_loss = train_and_benchmark(baseline_loader)
print(f"  Time: {baseline_time:.4f}s | Loss: {baseline_loss:.4f}")
prev_time = baseline_time


=== Progressive Optimization Results ===

Baseline (num_workers=0, pin_memory=False):
  Time: 33.9494s | Loss: 2.3171


Optimización del tamaño de lote
================================

El parámetro `batch_size` controla cuántas muestras se procesan juntas.
Elegir el tamaño de lote adecuado implica equilibrar varios factores:

**Consideraciones de memoria:**

-   Los tamaños de lote más grandes requieren más memoria GPU para almacenar
    entradas, activaciones y gradientes
-   Los errores de memoria insuficiente (OOM) son frecuentes con tamaños de
    lote grandes
-   Los tamaños moderados (32-128) suelen ofrecer el mejor equilibrio

**Dinámica del entrenamiento:**

-   El tamaño de lote afecta a la tasa de aprendizaje efectiva, por lo que
    generalmente requiere ajuste
-   Los lotes más grandes ofrecen estimaciones del gradiente más estables,
    pero pueden generalizar de forma diferente

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Al cambiar el tamaño de lote, recuerda ajustar los parámetros del optimizador,
especialmente el calendario de tasa de aprendizaje, a menos que estés haciendo inferencia.</p>

</div>

Dado que el tamaño de lote depende del modelo (no es una optimización que se
pueda añadir directamente), lo evaluamos de forma aislada en lugar de incluirlo
en la cadena de optimización progresiva.


In [12]:
# =============================================================================
# BENCHMARK DE TAMAÑO DE LOTE
#
# Evalúa el impacto del batch_size en el rendimiento de carga de datos.
# Se usa un dataset sin retardo para aislar el efecto del tamaño de lote
# de las transformaciones costosas.
#
# Lotes más grandes:
#   + Reducen el número de llamadas al DataLoader → menos overhead de IPC
#   + Mejor utilización del ancho de banda PCIe (menos transferencias pequeñas)
#   - Requieren más memoria GPU por paso
#   - Pueden necesitar ajuste de tasa de aprendizaje para converger bien
# =============================================================================

# Dataset sin transformaciones costosas: solo se mide el overhead de carga y transferencia
batch_dataset = SyntheticDatasetBatched(size=1000, transform_delay=0)


def benchmark_batch_size(batch_size, num_batches=10):
    """Benchmark de carga de datos para un tamaño de lote específico."""
    loader = DataLoader(batch_dataset, batch_size=batch_size, shuffle=True)
    start = time.perf_counter()
    for i, (data, labels) in enumerate(loader):
        if i >= num_batches:
            break
        data = data.to(device, non_blocking=True)  # Transferencia asíncrona CPU→GPU
        _ = data.sum()                             # Operación trivial para forzar que el tensor llegue a GPU
    if torch.cuda.is_available():
        # Espera a que todas las operaciones GPU terminen antes de medir el tiempo
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return elapsed


# Compara distintos tamaños de lote para identificar el óptimo en este hardware
print("\nBatch size comparison (isolated benchmark):")
for bs in [16, 32, 64, 128]:
    elapsed = benchmark_batch_size(bs)
    print(f"  Batch size {bs:3d}: {elapsed:.4f}s for 10 batches")



Batch size comparison (isolated benchmark):
  Batch size  16: 0.2416s for 10 batches
  Batch size  32: 0.4634s for 10 batches
  Batch size  64: 0.6837s for 10 batches
  Batch size 128: 0.8542s for 10 batches


Número de trabajadores (`num_workers`)
=======================================

El parámetro `num_workers` controla cuántos subprocesos se utilizan para la
carga de datos. Es crucial para paralelizar las transformaciones costosas.

**Cómo funciona:**

-   Cada trabajador mantiene una cola de lotes (controlada por `prefetch_factor`)
-   Los trabajadores preparan lotes en paralelo y los transfieren al proceso principal
-   Si `in_order=True` (por defecto), los lotes se devuelven en orden

**Cuándo aumentar `num_workers`:**

-   Cuando las transformaciones son computacionalmente costosas (augmentations, decodificación)
-   Cuando los datos se cargan desde almacenamiento lento (unidades de red, HDD)
-   Cuando se observa tiempo de inactividad de la GPU por la carga de datos

**Cuándo `num_workers=0` puede ser más rápido:**

-   Cuando las transformaciones son simples (operaciones básicas sobre tensores)
-   Cuando los datos ya están en memoria
-   Cuando la sobrecarga de la comunicación entre procesos (IPC) supera los
    beneficios de la paralelización

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Encontrar el valor óptimo de <code>num_workers</code> requiere ajuste: aumenta los
trabajadores hasta que el rendimiento se estabilice. Demasiados trabajadores
desperdician CPU y memoria (cada trabajador mantiene su propia copia del dataset y
los lotes precargados) y pueden causar el agotamiento de <code>/dev/shm</code>. Un buen
punto de partida son 2-4 trabajadores por GPU; perfilar con distintos valores
para encontrar el punto óptimo para tu carga de trabajo.</p>

</div>

Añadamos `num_workers=4` y `prefetch_factor=2` a nuestro bucle de entrenamiento
y midamos la mejora:


In [13]:
# =============================================================================
# OPTIMIZACIÓN 1: Multiprocesamiento con num_workers y prefetch_factor
#
# num_workers=4 lanza 4 subprocesos que preparan lotes en paralelo,
# solapando la carga de datos con el cómputo GPU del lote anterior.
#
# prefetch_factor=2 mantiene 2 lotes listos por trabajador en cola,
# actuando como búfer para absorber variaciones en la velocidad de carga.
# Con 4 workers y factor 2, puede haber hasta 8 lotes en memoria esperando.
# =============================================================================

workers_loader = DataLoader(
    benchmark_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,        # 4 subprocesos preparando lotes en paralelo
    prefetch_factor=2,    # 2 lotes precargados por trabajador (8 en total en memoria)
    pin_memory=False,     # Todavía sin memoria fijada (se añade en la siguiente optimización)
)

print("\n+ num_workers=4, prefetch_factor=2:")
workers_time, workers_loss = train_and_benchmark(workers_loader)
print(f"  Time: {workers_time:.4f}s | Loss: {workers_loss:.4f}")
print(
    f"  Speedup vs baseline: {baseline_time / workers_time:.2f}x | vs previous: {prev_time / workers_time:.2f}x"
)
# Actualiza el tiempo de referencia para comparar con la siguiente optimización
prev_time = workers_time



+ num_workers=4, prefetch_factor=2:
  Time: 12.5080s | Loss: 2.3169
  Speedup vs baseline: 2.71x | vs previous: 2.71x


Comprensión de `pin_memory`
============================

El parámetro `pin_memory` permite transferencias de datos de CPU a GPU más
rápidas mediante el uso de memoria fijada (page-locked).

**Cómo funciona la memoria fijada:**

-   La memoria fijada no puede intercambiarse al disco por el SO
-   Esto permite transferencias DMA (Acceso Directo a Memoria) más rápidas hacia la GPU
-   La transferencia de CPU a GPU puede realizarse de forma asíncrona

**Buenas prácticas:**

1.  Usa `pin_memory=True` en el DataLoader (enfoque recomendado)
2.  Combínalo con `non_blocking=True` al mover datos a la GPU
3.  Evita llamar manualmente a `tensor.pin_memory()` seguido de
    `.to(device, non_blocking=True)` — esto es más lento porque
    `pin_memory()` es bloqueante

**El patrón seguro:**

``` {.python}
# Recomendado: dejar que el DataLoader gestione el fijado
loader = DataLoader(dataset, pin_memory=True)
for data, labels in loader:
    data = data.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
```

::: {.seealso}
Para más detalles, consulta el [tutorial de pin\_memory](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)
:::

Añadamos `pin_memory=True` a nuestra configuración:


In [ ]:
# =============================================================================
# OPTIMIZACIÓN 2: Memoria fijada (pin_memory)
#
# pin_memory=True instruye al DataLoader a asignar los tensores del lote en
# memoria fijada (page-locked) del lado de la CPU. Esto habilita:
#   - DMA directo: la GPU puede leer la memoria sin intervención de la CPU
#   - Transferencias asíncronas: .to(device, non_blocking=True) no bloquea la CPU
#   - Mayor ancho de banda PCIe efectivo para la transferencia CPU→GPU
#
# NOTA: pin_memory solo tiene efecto con CUDA; se activa condicionalmente.
# =============================================================================

pinmem_loader = DataLoader(
    benchmark_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    prefetch_factor=2,
    pin_memory=torch.cuda.is_available(),  # Activa solo si hay GPU CUDA disponible
)

if torch.cuda.is_available():
    print("\n+ pin_memory=True:")
    pinmem_time, pinmem_loss = train_and_benchmark(pinmem_loader)
    print(f"  Time: {pinmem_time:.4f}s | Loss: {pinmem_loss:.4f}")
    print(
        f"  Speedup vs baseline: {baseline_time / pinmem_time:.2f}x | vs previous: {prev_time / pinmem_time:.2f}x"
    )
    print(
        "  (pin_memory benefit is modest here because CPU transform time dominates H2D transfer)"
    )
    # Actualiza el tiempo de referencia para comparar con la siguiente optimización
    prev_time = pinmem_time
else:
    print("\n+ pin_memory: skipped (CUDA not available)")
    pinmem_time = workers_time



+ pin_memory=True:
  Time: 12.5694s | Loss: 2.3209
  Speedup vs baseline: 2.70x | vs previous: 1.00x
  (pin_memory benefit is modest here because CPU transform time dominates H2D transfer)


Trabajadores persistentes
==========================

Por defecto, los procesos trabajadores se apagan y reinician entre épocas.
Esto genera una sobrecarga de arranque (importación de módulos, creación de
procesos, reinicialización de datasets) en cada límite de época.

Establecer `persistent_workers=True` mantiene los trabajadores activos entre
épocas, eliminando este coste repetido de arranque.

**Cuándo ayuda más:**

-   Al entrenar durante muchas épocas con datasets pequeños
-   Cuando el `__init__` del dataset es costoso (p. ej., carga de metadatos)
-   Cuando se combina con un valor alto de `num_workers`

Comparemos con y sin trabajadores persistentes a lo largo de múltiples épocas:


In [ ]:
# =============================================================================
# OPTIMIZACIÓN 3: Trabajadores persistentes (persistent_workers)
#
# Sin persistent_workers=True (comportamiento por defecto), los subprocesos
# trabajadores del DataLoader se destruyen al final de cada época y se crean
# de nuevo al inicio de la siguiente. Este ciclo tiene un coste real:
#   - Inicio de nuevos procesos Python
#   - Re-importación de módulos en cada trabajador
#   - Re-inicialización del dataset en cada trabajador
#
# Con persistent_workers=True, los trabajadores permanecen activos entre
# épocas: solo esperan inactivos hasta que se necesiten de nuevo. El ahorro
# es especialmente notable con muchas épocas y datasets costosos de inicializar.
# =============================================================================

# Loader SIN trabajadores persistentes: reinicia subprocesos en cada época
non_persistent_loader = DataLoader(
    benchmark_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    prefetch_factor=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=False,   # Por defecto: reinicia trabajadores en cada época
)

# Loader CON trabajadores persistentes: mantiene los subprocesos activos entre épocas
persistent_loader = DataLoader(
    benchmark_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    prefetch_factor=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,    # Mantiene trabajadores entre épocas → elimina el coste de reinicio
)

print("\n+ persistent_workers=True (10 epochs):")
non_persistent_time, _ = train_and_benchmark(non_persistent_loader)       # Tiempo con reinicio de workers
persistent_time, persistent_loss = train_and_benchmark(persistent_loader)  # Tiempo con workers persistentes
print(f"  Without persistent_workers: {non_persistent_time:.4f}s")
print(f"  With persistent_workers:    {persistent_time:.4f}s")
print(
    f"  Speedup vs baseline: {baseline_time / persistent_time:.2f}x | vs previous: {prev_time / persistent_time:.2f}x"
)
# Actualiza el tiempo de referencia para comparar con la siguiente optimización
prev_time = persistent_time


+ persistent_workers=True (10 epochs):
  Without persistent_workers: 12.6166s
  With persistent_workers:    11.2962s
  Speedup vs baseline: 3.01x | vs previous: 1.11x


In [16]:
# =============================================================================
# OPTIMIZACIÓN 4: USO DEL DataPrefetcher EN EL BUCLE DE ENTRENAMIENTO
# (se integra vía el parámetro prefetch_device en train_and_benchmark)
# =============================================================================

if torch.cuda.is_available():
    print("\n+ DataPrefetcher (overlapping H2D transfer):")
    prefetch_time, prefetch_loss = train_and_benchmark(
        persistent_loader, prefetch_device=device  # Activa el prefetching H2D
    )
    print(f"  Time: {prefetch_time:.4f}s | Loss: {prefetch_loss:.4f}")
    print(
        f"  Speedup vs baseline: {baseline_time / prefetch_time:.2f}x | vs previous: {prev_time / prefetch_time:.2f}x"
    )
    # Actualiza el tiempo de referencia para comparar con la siguiente optimización
    prev_time = prefetch_time
else:
    print("\n+ DataPrefetcher: skipped (CUDA not available)")
    prefetch_time = persistent_time



+ DataPrefetcher (overlapping H2D transfer):
  Time: 11.2031s | Loss: 2.3194
  Speedup vs baseline: 3.03x | vs previous: 1.01x


Optimización a nivel de dataset: `__getitems__`
================================================

Más allá de ajustar los parámetros del DataLoader, puedes optimizar el propio
dataset. El DataLoader de PyTorch soporta un protocolo de obtención por lotes
mediante `__getitems__`: si tu dataset define este método, el cargador lo llama
una sola vez con una lista de índices en lugar de llamar a `__getitem__`
repetidamente para cada muestra.

**Cómo funciona:**

-   El cargador por defecto hace: `[dataset[idx] for idx in batch_indices]`
-   Con `__getitems__`: `dataset.__getitems__(batch_indices)`

**Cuándo ayuda:**

-   Cuando la sobrecarga por muestra es significativa (p. ej., abrir conexiones,
    parsear cabeceras, adquirir bloqueos)
-   Cuando los datos pueden obtenerse en bloque de forma más eficiente (p. ej.,
    una consulta SQL para N filas en lugar de N consultas, o generación vectorizada
    de tensores)
-   Cuando la transformación tiene un coste de inicialización fijo que puede
    amortizarse sobre el lote

**Firma esperada:**

``` {.python}
def __getitems__(self, indices: list[int]) -> list:
    # Obtiene todos los elementos a la vez y los devuelve como lista
    ...
```

Nuestro `SyntheticDatasetBatched` implementa `__getitems__` para generar el
lote completo en una sola llamada vectorizada (con un único retardo amortizado)
en lugar de N llamadas individuales cada una con su propio retardo. Añadamos
esto a nuestra configuración acumulada:


In [ ]:
# =============================================================================
# OPTIMIZACIÓN 5: __getitems__ — Obtención vectorizada por lotes
#
# Con __getitems__, el DataLoader llama UNA SOLA VEZ al dataset con todos los
# índices del lote en lugar de N veces a __getitem__. Esto permite:
#   - Un único retardo (o una sola operación de I/O) por lote en vez de N
#   - Generación vectorizada de tensores para todo el lote a la vez
#
# En SyntheticDatasetBatched el efecto es dramático: en vez de 32 × 5 ms = 160 ms
# de retardo acumulado por lote, se incurre en solo 5 ms para el lote completo.
# En datasets reales equivale a: una consulta SQL para N filas, una apertura
# de archivo seguida de N lecturas, o un decode vectorizado con OpenCV/PIL.
# =============================================================================

# Dataset con __getitems__: igual configuración que benchmark_dataset pero
# usando SyntheticDatasetBatched para aprovechar la obtención vectorizada
benchmark_dataset_batched = SyntheticDatasetBatched(
    size=512, feature_dim=224, transform_delay=0.005
)

# Loader con todas las optimizaciones acumuladas + __getitems__
batched_loader = DataLoader(
    benchmark_dataset_batched,
    batch_size=32,
    shuffle=True,
    num_workers=4,             # Multiprocesamiento paralelo
    prefetch_factor=2,         # Precarga de 2 lotes por trabajador
    pin_memory=torch.cuda.is_available(),  # Memoria fijada para DMA
    persistent_workers=True,   # Trabajadores persistentes entre épocas
    # __getitems__ se activa automáticamente si el dataset lo define
)

print("\n+ __getitems__ (batched dataset fetching):")
batched_time, batched_loss = train_and_benchmark(batched_loader)
print(f"  Time: {batched_time:.4f}s | Loss: {batched_loss:.4f}")
print(
    f"  Speedup vs baseline: {baseline_time / batched_time:.2f}x | vs previous: {prev_time / batched_time:.2f}x"
)
# Actualiza la referencia para posibles comparaciones posteriores
prev_time = batched_time


+ __getitems__ (batched dataset fetching):
  Time: 2.6616s | Loss: 2.3205
  Speedup vs baseline: 12.76x | vs previous: 4.21x


Parámetro `in_order`
=====================

Por defecto (`in_order=True`), el DataLoader devuelve los lotes en el mismo
orden que los índices del dataset. Esto requiere almacenar en caché los lotes
que llegan fuera de orden desde los trabajadores.

**Cuándo considerar `in_order=False`:**

-   Cuando no necesitas un orden determinista (p. ej., sin checkpointing)
-   Cuando observas picos de entrenamiento debidos al almacenamiento en caché
    de lotes
-   Cuando maximizar el rendimiento es más importante que la reproducibilidad

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p><code>in_order=False</code> puede no aumentar el rendimiento medio, pero puede
reducir la varianza y eliminar lotes ocasionalmente lentos causados por el
bloqueo de cabecera de cola cuando un trabajador es más lento que los demás.</p>

</div>



Frecuencia de instantáneas (`snapshot_every_n_steps`)
======================================================

Al usar el StatefulDataLoader de torchdata (para checkpointing), el parámetro
`snapshot_every_n_steps` controla con qué frecuencia se guarda el estado del
DataLoader.

**Compromisos:**

-   **Mayor frecuencia (n pequeño):** más sobrecarga, pero menor pérdida de datos
    ante un fallo del proceso
-   **Menor frecuencia (n grande):** menos sobrecarga, pero más muestras a
    reprocesar al recuperarse

Elige en función de tus requisitos de tolerancia a fallos y del coste de
reprocesar los datos.


Memoria compartida y `set_sharing_strategy`
============================================

Al usar multiprocesamiento con `num_workers > 0`, PyTorch necesita transferir
tensores entre los procesos trabajadores y el proceso principal. La estrategia
de compartición determina cómo se hace esto.

**Estrategias disponibles:**

PyTorch ofrece dos estrategias de compartición mediante
`torch.multiprocessing.set_sharing_strategy()`:

1.  **file\_descriptor** (por defecto en la mayoría de sistemas)
    -   Usa descriptores de archivo para compartir memoria
    -   Limitada por el límite de descriptores de archivo del sistema (`ulimit -n`)
    -   Más eficiente para tensores pequeños
2.  **file\_system**
    -   Usa archivos de memoria compartida en `/dev/shm`
    -   No limitada por el número de descriptores de archivo
    -   Mejor para grandes cantidades de tensores
    -   Bajo coste de transformación


**Cómo cambiar la estrategia:**

``` {.python}
import torch.multiprocessing as mp

# Cambiar a la estrategia file_system
# Debe llamarse antes de crear cualquier trabajador de DataLoader
mp.set_sharing_strategy('file_system')
```

**Cómo elegir la estrategia adecuada:**

  ------------------------------------------------------------------------
  Escenario               Estrategia recomendada          Razón
  ----------------------- ------------------------------- ----------------
  Muchos tensores pequeños  file\_descriptor (por defecto)  Menor sobrecarga por tensor

  Pocos tensores grandes    file\_system                    Evita límites de fd

  num\_workers alto         file\_system                    Evita el agotamiento de fd
  ------------------------------------------------------------------------

<div style="background-color: #e94f3b; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>ADVERTENCIA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p><code>set_sharing_strategy()</code> debe llamarse antes de crear cualquier
DataLoader con <code>num_workers &gt; 0</code>. Cambiarlo después no tiene efecto
sobre los trabajadores ya existentes.</p>

</div>



Gestión de memoria compartida insuficiente (`/dev/shm`)
========================================================

Al usar `num_workers > 0`, PyTorch utiliza memoria compartida (`/dev/shm`) para
pasar datos eficientemente entre los procesos trabajadores y el proceso principal.
Si encuentras errores como:

``` {.text}
RuntimeError: unable to open shared memory object </torch_xxx>
ERROR: Unexpected bus error encountered in worker
```

Esto normalmente significa que has agotado la asignación de memoria compartida.

**Soluciones:**

**1. Aumentar el tamaño de /dev/shm (si es posible)**

**2. Reducir la presión de memoria del DataLoader:**

``` {.python}
# Reducir el número de trabajadores
DataLoader(dataset, num_workers=2)  # En lugar de 8+

# Reducir el factor de precarga
DataLoader(dataset, num_workers=4, prefetch_factor=1)  # En lugar de 2

# Usar tamaños de lote más pequeños
DataLoader(dataset, batch_size=16)  # Lotes más pequeños = menos shm
```

**3. Cambiar la estrategia de compartición:**

``` {.python}
import torch.multiprocessing as mp
mp.set_sharing_strategy('file_system')
```

**4. Limpiar memoria compartida con fugas:**

``` {.bash}
# Listar segmentos de memoria compartida
ls -la /dev/shm/

# Eliminar segmentos de PyTorch huérfanos (¡con precaución!)
rm /dev/shm/torch_*
```

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Las fugas de memoria compartida pueden ocurrir si los procesos trabajadores
se interrumpen sin limpieza adecuada.</p>

</div>



Resumen final
=============

Aquí está el efecto acumulado de cada optimización aplicada a nuestro bucle de
entrenamiento. Cada fila incluye todas las optimizaciones de las filas anteriores:

::: {.rst-class}
summary-table
:::

  -----------------------------------------------------------------------
  Configuración                                    vs Base        vs Anterior
  ------------------------------------------------ -------------- --------------
  Base (num\_workers=0, sin fijado)                1.00x          ---

  \+ num\_workers=4, prefetch\_factor=2            \~2.7x         \~2.7x

  \+ pin\_memory=True                              \~2.8x         \~1.0x

  \+ persistent\_workers=True                      \~3.7x         \~1.3x

  \+ DataPrefetcher (solapamiento H2D)             \~3.6x         \~1.0x

  \+ \_\_getitems\_\_ (obtención por lotes)        \~10x          \~2.9x
  -----------------------------------------------------------------------

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Estos resultados están basados en nuestro dataset de benchmark.
Las aceleraciones reales variarán según tu carga de trabajo específica,
hardware, tamaño del dataset y complejidad de las transformaciones.</p>

</div>



Resumen y buenas prácticas
===========================

1.  **Comienza con tamaños de lote moderados** (32-128) y escala hacia arriba
    si la memoria lo permite.
2.  **Usa `num_workers > 0`** cuando las transformaciones sean costosas.
    Empieza con 2-4 trabajadores y aumenta según la capacidad de memoria.
    Más no siempre es mejor.
3.  **Activa `pin_memory=True`** cuando uses un acelerador.
4.  **Usa `persistent_workers=True`** para evitar la sobrecarga de reinicio
    de trabajadores entre épocas.
5.  **Perfilar tu pipeline** para identificar cuellos de botella en la CPU
    durante el acceso al dataset, las transformaciones, etc.
6.  **Implementar precarga de datos** para cargas de trabajo en GPU y solapar
    la transferencia de datos con el cómputo.
7.  **Usa la estrategia de compartición `file_system`** cuando alcances los
    límites de descriptores de archivo.


Conclusión
==========

En este tutorial hemos aprendido a optimizar progresivamente un pipeline de
carga de datos en PyTorch — desde un punto de partida naive con un único proceso
hasta una configuración completamente optimizada que usa trabajadores con
multiprocesamiento, memoria fijada, trabajadores persistentes, precarga basada en
streams CUDA y obtención de datos por lotes con `__getitems__`. Cada optimización
aborda un cuello de botella diferente y, juntas, pueden suponer una mejora de un
orden de magnitud en el rendimiento. Estas deben considerarse buenas prácticas,
y el rendimiento depende de la carga de trabajo específica.


Recursos adicionales
====================

-   [Documentación de PyTorch DataLoader](https://pytorch.org/docs/stable/data.html)
-   [Tutorial de Pin Memory y transferencia Non-blocking](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)
-   [Guía de ajuste de rendimiento de PyTorch](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html)
